# Toto-2.0-FnF on GIFT-Eval

FFORMA-style XGBoost meta-learner that gates a pool of foundation models (Toto 2.0 family + open foundation models: Chronos-2, TimesFM-2.5, FlowState, TiRex, PatchTST-FM). Each (frequency, term) bucket has its own XGBoost head that outputs softmax weights over the candidate models for every test window; the final forecast is the weight-summed quantile prediction of the base models.

This notebook replicates the submission. The fast path (the default below) downloads the trained booster manifest + per-model test predictions + per-window time-series features from a Hugging Face dataset repo and runs Inference + evaluation locally — no GPU required. Optional sections show how to regenerate the artifacts from scratch.

## Setup

1. Install GIFT-Eval as instructed in the [README](../README.md).
2. Download the GIFT-Eval dataset:
   ```bash
   huggingface-cli download Salesforce/GiftEval --repo-type=dataset --local-dir /path/to/gifteval
   ```
3. Install ensemble dependencies:
   ```bash
   pip install xgboost huggingface_hub fsspec
   ```
4. Set the `GIFT_EVAL` env var to the dataset path (see the cell below).

In [ ]:
import os
import json
import base64
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from gluonts.ev.metrics import (
    MAE, MAPE, MASE, MSE, MSIS, ND, NRMSE, RMSE, SMAPE,
    MeanWeightedSumQuantileLoss,
)
from gluonts.model import evaluate_model
from gluonts.model.forecast import QuantileForecast
from gluonts.model.predictor import RepresentablePredictor
from gluonts.time_feature import get_seasonality
from huggingface_hub import snapshot_download

from gift_eval.data import Dataset

os.environ["GIFT_EVAL"] = "Change/To/GiftEval/Local/Path"

In [ ]:
MODEL_NAME = "Toto-2.0-FnF"

# Base-model pool. Order must match the meta-learner booster's class indices —
# do NOT reorder unless you also retrain the booster.
MODELS = [
    "chronos-2",
    "timesfm-2.5",
    "flowstate",
    "tirex",
    "patchtst-fm",
    "toto-2.0-4m",
    "toto-2.0-22m",
    "toto-2.0-313m",
    "toto-2.0-1b",
    "toto-2.0-2.5b",
]

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# HF dataset repo holding the artifact bundle (booster + features + per-model
# predictions). See the bottom of this notebook for the layout + upload script.
HF_ARTIFACTS_REPO = "Datadog/Toto-2.0-Family-and-Friends"
ARTIFACTS_DIR = Path("./toto_2_0_fnf_artifacts")

PRETTY_DATASET_NAMES = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}

SHORT_DATASETS = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H".split()
MED_LONG_DATASETS = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H".split()
ALL_DATASETS = sorted(set(SHORT_DATASETS) | set(MED_LONG_DATASETS))

In [ ]:
METRICS = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(forecast_type="mean"),
    NRMSE(forecast_type="mean"),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=QUANTILE_LEVELS),
]

METRIC_COLUMNS = [
    "MSE[mean]", "MSE[0.5]", "MAE[0.5]", "MASE[0.5]", "MAPE[0.5]",
    "sMAPE[0.5]", "MSIS", "RMSE[mean]", "NRMSE[mean]", "ND[0.5]",
    "mean_weighted_sum_quantile_loss",
]

## 1. Download artifacts from Hugging Face Hub

This is the fast path. We host the trained meta-learner boosters, per-(dataset, term) time-series features for the test split, and per-(model, dataset, term) quantile predictions on a private HF model repo. Downloading them lets you reproduce our submission without retraining the meta-learner or re-running every base model.

In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=HF_ARTIFACTS_REPO,
    repo_type="model",
    local_dir=str(ARTIFACTS_DIR),
    # On reruns, only download files we don't have or that changed.
    local_dir_use_symlinks=False,
)

# Bundle layout:
#   booster_manifest.json   — base64-encoded XGBoost boosters keyed by "freq|term"
#   feature_columns.json    — train-time column order the boosters expect
#   categories.json         — train-time categorical sets for `freq`, `domain`
#   models.json             — model list in column order (sanity check vs MODELS above)
#   test_features/<ds_dirname>/test_features.npz + test_metadata.npz
#   test_predictions/<model>/<ds_dirname>/test_predictions.npz

with open(ARTIFACTS_DIR / "models.json") as f:
    bundle_models = json.load(f)
assert bundle_models == MODELS, (
    f"Bundle model order {bundle_models} does not match MODELS {MODELS}; "
    f"the booster's class indices are tied to this order."
)

with open(ARTIFACTS_DIR / "feature_columns.json") as f:
    FEATURE_COLUMNS = json.load(f)
with open(ARTIFACTS_DIR / "categories.json") as f:
    CATEGORIES = json.load(f)

SCALAR_FEATURE_COLUMNS = ["seasonality", "prediction_length", "num_variates"]
CATEGORICAL_FEATURE_COLUMNS = ["freq", "domain"]
CANONICAL_TS_FEATURES = [
    c for c in FEATURE_COLUMNS
    if c not in SCALAR_FEATURE_COLUMNS + CATEGORICAL_FEATURE_COLUMNS
]
print(f"Loaded {len(FEATURE_COLUMNS)} feature columns; "
      f"|freq|={len(CATEGORIES['freq'])}, |domain|={len(CATEGORIES['domain'])}")

In [ ]:
# Load the per-bucket booster manifest. Keys are "freq|term" strings; freq is
# canonical (anchor suffix stripped — W-TUE -> W, Q-DEC -> Q).
with open(ARTIFACTS_DIR / "booster_manifest.json") as f:
    manifest = json.load(f)

BOOSTERS: dict[tuple[str, str], xgb.Booster] = {}
for key, b64 in manifest.items():
    freq, term = key.split("|", 1)
    booster = xgb.Booster()
    booster.load_model(bytearray(base64.b64decode(b64)))
    BOOSTERS[(freq, term)] = booster
print(f"Loaded {len(BOOSTERS)} per-(freq, term) meta-learner boosters")


def canonical_freq(freq: str) -> str:
    """Strip pandas anchor suffix so W-TUE, W-FRI, W-SUN -> W; Q-DEC -> Q. The
    raw freq with anchor is still kept as a categorical feature for XGBoost
    — this is only for the bucket lookup key."""
    return freq.split("-", 1)[0]

## 2. Inference

All ensemble inference work — feature-frame construction, softmax weighting, dropping models with missing predictions and renormalizing, weight-summing across the model axis — lives in a custom gluonts predictor (`Toto2FnFPredictor`). The driver loop then mirrors every other GIFT-Eval submission: instantiate one predictor per (dataset, term), call `gluonts.evaluate_model` with the standard arguments.

Per (dataset, term) the predictor:
1. Loads `test_features.npz` + `test_metadata.npz` for the current dataset; reindexes the tsfeatures against `FEATURE_COLUMNS` and attaches the categorical/scalar features using the train-time category sets in `CATEGORIES`.
2. Predicts raw class logits from the `(canonical_freq, term)` bucket booster → softmax → per-window weights.
3. Stacks each base model's pre-computed quantile predictions, drops any model whose `test_predictions.npz` is missing, renormalizes the weights per row.
4. Weight-sums across the model axis → final per-window quantile forecast yielded as a `QuantileForecast` from `predict()`.

In [ ]:
def _load_npz(path: Path) -> dict | None:
    if not path.exists():
        return None
    with np.load(path) as npz:
        return {k: npz[k] for k in npz.files}


def _scalar(arr):
    a = np.asarray(arr)
    return a.item() if a.shape == () else a


class Toto2FnFPredictor(RepresentablePredictor):
    """gluonts predictor that softmax-gates a pool of base models via a tuned
    XGBoost meta-learner and ensembles pre-computed per-model quantile
    forecasts on the fly.

    All ensemble inference work (feature frame construction, softmax weighting,
    per-window ensembling, dropping models with missing predictions and
    renormalizing) is encapsulated here so the driver loop only has to call
    ``gluonts.evaluate_model(predictor, test_data=...)`` — same call shape
    every other GIFT-Eval submission uses.

    Construct one predictor per (dataset, term); the heavy ensemble tensor
    is built lazily on the first ``predict()`` call and cached.
    """

    def __init__(
        self,
        prediction_length: int,
        *,
        ds_dirname: str,
        booster: xgb.Booster,
        artifacts_dir: Path,
        models: list[str],
        feature_columns: list[str],
        canonical_ts_features: list[str],
        categorical_columns: list[str],
        categories: dict,
        quantile_levels: list[float],
    ):
        super().__init__(prediction_length=prediction_length)
        self._ds_dirname = ds_dirname
        self._booster = booster
        self._artifacts_dir = artifacts_dir
        self._models = models
        self._feature_columns = feature_columns
        self._canonical_ts_features = canonical_ts_features
        self._categorical_columns = categorical_columns
        self._categories = categories
        self._quantile_keys = [str(q) for q in quantile_levels]
        self._ensemble = None

    def _build_feature_frame(self, feats: dict, meta: dict) -> pd.DataFrame:
        # Reindex this dataset's tsfeatures against the train-time canonical
        # set; columns not produced for this dataset (e.g. seasonal_strength
        # on yearly data) become NaN, which XGBoost handles natively.
        df = (
            pd.DataFrame(feats["X"], columns=[str(n) for n in feats["feature_names"]])
            .reindex(columns=self._canonical_ts_features)
            .astype(np.float32)
        )
        df["seasonality"] = np.float32(int(_scalar(meta["seasonality"])))
        df["prediction_length"] = np.float32(int(_scalar(meta["prediction_length"])))
        df["num_variates"] = np.float32(int(_scalar(meta["num_variates"])))
        df["freq"] = str(_scalar(meta["freq"]))
        df["domain"] = str(_scalar(meta["domain"]))
        df = df.reindex(columns=self._feature_columns)
        for col in self._categorical_columns:
            df[col] = df[col].astype(pd.CategoricalDtype(categories=self._categories[col]))
        return df

    def _build_ensemble(self) -> np.ndarray:
        feats = _load_npz(self._artifacts_dir / "test_features" / self._ds_dirname / "test_features.npz")
        meta = _load_npz(self._artifacts_dir / "test_features" / self._ds_dirname / "test_metadata.npz")
        if feats is None or meta is None:
            raise FileNotFoundError(f"missing test features for {self._ds_dirname}")

        df = self._build_feature_frame(feats, meta)
        n = len(df)

        # Predict raw class logits → softmax → per-window weights.
        raw = self._booster.predict(
            xgb.DMatrix(df, enable_categorical=True), output_margin=True,
        ).reshape(n, len(self._models))
        z = raw - raw.max(axis=1, keepdims=True)
        e = np.exp(z)
        w = e / e.sum(axis=1, keepdims=True)

        # Stack per-model predictions in MODELS order; drop missing and
        # renormalize the remaining weights per row.
        per_model_preds, valid_mask = [], []
        for model in self._models:
            npz = _load_npz(self._artifacts_dir / "test_predictions" / model / self._ds_dirname / "test_predictions.npz")
            if npz is None or "predictions" not in npz:
                per_model_preds.append(None)
                valid_mask.append(False)
                continue
            per_model_preds.append(npz["predictions"].astype(np.float32))
            valid_mask.append(True)
        if not any(valid_mask):
            raise RuntimeError(f"no base model has predictions for {self._ds_dirname}")

        valid_idx = np.array([i for i, ok in enumerate(valid_mask) if ok])
        w_valid = w[:, valid_idx]
        w_valid = w_valid / w_valid.sum(axis=1, keepdims=True).clip(min=1e-12)
        pred_stack = np.stack([per_model_preds[i] for i in valid_idx], axis=1)
        # pred_stack: (n, M_valid, Q, T); ensemble: (n, Q, T)
        return (w_valid[:, :, None, None] * pred_stack).sum(axis=1)

    def predict(self, dataset, **kwargs):
        if self._ensemble is None:
            self._ensemble = self._build_ensemble()
        for i, entry in enumerate(dataset):
            yield QuantileForecast(
                forecast_arrays=self._ensemble[i],
                forecast_keys=self._quantile_keys,
                start_date=entry["start"] + len(entry["target"]),
                item_id=entry.get("item_id"),
            )

In [ ]:
def evaluate_dataset(dataset_name: str, term: str, properties: dict) -> dict | None:
    # `to_univariate=True` for a target_dim==1 dataset triggers a gift_eval
    # bug (MultivariateToUnivariate iterates each scalar value instead of
    # each variate, producing 0-d targets that crash gluonts' test_pair
    # offset calc). Probe target_dim with to_univariate=False first and
    # only switch to True when actually multivariate. For target_dim==1
    # the two iteration orders are equivalent, so the saved
    # test_predictions.npz still aligns row-for-row with `test_data`.
    probe = Dataset(
        storage_env_var="GIFT_EVAL",
        name=dataset_name, term=term, to_univariate=False,
    )
    ds = probe if probe.target_dim == 1 else Dataset(
        storage_env_var="GIFT_EVAL",
        name=dataset_name, term=term, to_univariate=True,
    )

    if "/" in dataset_name:
        ds_key = PRETTY_DATASET_NAMES.get(dataset_name.split("/")[0].lower(),
                                          dataset_name.split("/")[0].lower())
    else:
        ds_key = PRETTY_DATASET_NAMES.get(dataset_name.lower(), dataset_name.lower())
    ds_freq = str(ds.freq)
    ds_config = f"{ds_key}/{ds_freq}/{term}"
    ds_dirname = ds_config.replace("/", "_")

    bucket_key = (canonical_freq(ds_freq), term)
    booster = BOOSTERS.get(bucket_key)
    if booster is None:
        print(f"  {ds_config}: no booster for bucket {bucket_key} — skipping")
        return None

    predictor = Toto2FnFPredictor(
        prediction_length=int(ds.prediction_length),
        ds_dirname=ds_dirname,
        booster=booster,
        artifacts_dir=ARTIFACTS_DIR,
        models=MODELS,
        feature_columns=FEATURE_COLUMNS,
        canonical_ts_features=CANONICAL_TS_FEATURES,
        categorical_columns=CATEGORICAL_FEATURE_COLUMNS,
        categories=CATEGORIES,
        quantile_levels=QUANTILE_LEVELS,
    )

    try:
        res = evaluate_model(
            predictor,
            test_data=ds.test_data,
            metrics=METRICS,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=int(get_seasonality(ds.freq)),
        )
    except Exception as e:
        print(f"  {ds_config}: FAILED: {e}")
        return None

    row = {"dataset": ds_config, "model": MODEL_NAME}
    for col in METRIC_COLUMNS:
        row[f"eval_metrics/{col}"] = res[col].iloc[0]
    row["domain"] = properties[ds_key]["domain"]
    row["num_variates"] = properties[ds_key]["num_variates"]
    print(f"  {ds_config}: MASE={row['eval_metrics/MASE[0.5]']:.4f}")
    return row

In [ ]:
properties = json.load(open("dataset_properties.json"))

rows = []
for dataset_name in ALL_DATASETS:
    for term in ["short", "medium", "long"]:
        if term in ("medium", "long") and dataset_name not in MED_LONG_DATASETS:
            continue
        row = evaluate_dataset(dataset_name, term, properties)
        if row is not None:
            rows.append(row)

results = pd.DataFrame(rows)
results

## 3. Save results for submission

In [ ]:
out_dir = Path(f"../results/{MODEL_NAME}")
out_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(out_dir / "all_results.csv", index=False)

config = {
    "model": MODEL_NAME,
    "model_type": "pretrained",
    "model_dtype": "float32",
    "model_link": f"https://huggingface.co/{HF_ARTIFACTS_REPO}",
    "code_link": "https://github.com/SalesforceAIResearch/gift-eval/blob/main/notebooks/toto_2_0_fnf.ipynb",
    "org": "Datadog",
    "testdata_leakage": "No",
    "replication_code_available": "Yes",
}
with open(out_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)
print(f"Wrote {out_dir / 'all_results.csv'} ({len(results)} rows) and config.json")

---

## Optional: regenerate artifacts from scratch

The fast path above already produces `all_results.csv` from the prebuilt bundle. The sections below are for full reproducibility — for each artifact in the bundle, they show how to regenerate it and verify against the published file. Skip them if you only want to score the submission.

### Optional A. Regenerate tsfeatures

Per-window time-series features come from the [`tsfeatures`](https://github.com/Nixtla/tsfeatures) library. Each window's feature vector is computed from the lookback context that precedes the forecast — `ds.test_data.input` yields the model inputs only, so this regenerator can't accidentally look at any ground-truth label.

In [ ]:
# pip install tsfeatures
from tsfeatures import tsfeatures


def _feature_context_cap(season: int) -> int:
    # Bound the per-window cost of STL/KPSS/Hurst/Holt-style features.
    return max(2048, season * 32)


def _build_panel(contexts, start) -> pd.DataFrame:
    pieces = []
    for i, ctx in enumerate(contexts):
        pieces.append(pd.DataFrame({
            "unique_id": i,
            "ds": pd.date_range(start.to_timestamp(), periods=len(ctx), freq=start.freq),
            "y": ctx,
        }))
    return pd.concat(pieces, ignore_index=True)


def regenerate_features(dataset_name: str, term: str, out_root: Path, threads: int = 4) -> str:
    """Recompute test_features.npz + test_metadata.npz for one (dataset, term).
    Returns the ds_dirname so the verifier can find the bundle counterpart.
    """
    probe = Dataset(storage_env_var="GIFT_EVAL", name=dataset_name, term=term, to_univariate=False)
    ds = probe if probe.target_dim == 1 else Dataset(
        storage_env_var="GIFT_EVAL", name=dataset_name, term=term, to_univariate=True,
    )
    season = int(get_seasonality(ds.freq))
    cap = _feature_context_cap(season)
    contexts, starts = [], []
    for inp in ds.test_data.input:
        ctx = np.asarray(inp["target"], dtype=np.float64).ravel()
        contexts.append(ctx[-cap:] if ctx.size > cap else ctx)
        starts.append(inp["start"])
    if not contexts:
        raise RuntimeError(f"no test windows for {dataset_name}/{term}")

    panel = _build_panel(contexts, starts[0])
    feats = (
        tsfeatures(panel, freq=season, threads=threads)
        .sort_values("unique_id").set_index("unique_id")
    )

    ds_key = PRETTY_DATASET_NAMES.get(
        dataset_name.split("/")[0].lower() if "/" in dataset_name else dataset_name.lower(),
        dataset_name.split("/")[0].lower() if "/" in dataset_name else dataset_name.lower(),
    )
    ds_dirname = f"{ds_key}_{str(ds.freq)}_{term}".replace("/", "_")
    out = out_root / ds_dirname
    out.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        out / "test_features.npz",
        X=feats.to_numpy(dtype=np.float32),
        feature_names=np.array(feats.columns.tolist()),
    )
    np.savez_compressed(
        out / "test_metadata.npz",
        seasonality=np.int32(season),
        prediction_length=np.int32(ds.prediction_length),
        num_variates=np.int32(getattr(ds, "target_dim", 1)),
        freq=np.array(str(ds.freq)),
        domain=np.array(properties.get(ds_key, {}).get("domain", "")),
    )
    return ds_dirname


def verify_features_against_bundle(ds_dirname: str, regen_root: Path, atol: float = 1e-5) -> None:
    """Assert regenerated test_features.npz matches the bundle's copy."""
    new_path = regen_root / ds_dirname / "test_features.npz"
    bundle_path = ARTIFACTS_DIR / "test_features" / ds_dirname / "test_features.npz"
    with np.load(new_path) as a, np.load(bundle_path) as b:
        assert list(a["feature_names"]) == list(b["feature_names"]), "column-name mismatch"
        diff = np.nan_to_num(a["X"] - b["X"], nan=0.0)
        max_abs = float(np.max(np.abs(diff)))
        assert max_abs <= atol, f"max abs diff {max_abs:.4g} > {atol}"
    print(f"  features for {ds_dirname}: match bundle (atol={atol})")

In [ ]:
# Smoke-test the feature regen on one small dataset.
REGEN_FEATURES_ROOT = Path("./regen/test_features")
ds_dirname = regenerate_features("m4_weekly", "short", REGEN_FEATURES_ROOT)
verify_features_against_bundle(ds_dirname, REGEN_FEATURES_ROOT)

### Optional B. Regenerate per-model test predictions

Each base model's predictions are stored as `(n_windows, 9, prediction_length)` float32 quantile arrays. Below is a unified `regenerate_predictions` driver that wraps any gluonts predictor, followed by one factory per pool member showing how to construct that predictor. Each factory mirrors the model's standalone GIFT-Eval notebook — install the model's package, load the checkpoint, wrap into a gluonts predictor with quantile output.

Foundation-model inference is GPU-intensive; running all 10 over all 97 (dataset, term) configs is a multi-hour job and may require a machine with enough VRAM for the largest checkpoints. The smoke test at the end runs each available factory on `m4_weekly/W/short` only and asserts the output matches the bundle within FP tolerance.

In [ ]:
def regenerate_predictions(predictor, model_name: str, dataset_name: str, term: str,
                          out_root: Path, batch_size: int = 512) -> tuple[str, np.ndarray]:
    """Run a gluonts predictor on the test split, stack quantile forecasts into
    a (n_windows, 9, prediction_length) array, and save to test_predictions.npz.

    The predictor must yield Forecast objects whose `.quantile(q)` returns an
    array of shape (prediction_length,) for each q in QUANTILE_LEVELS.
    """
    probe = Dataset(storage_env_var="GIFT_EVAL", name=dataset_name, term=term, to_univariate=False)
    ds = probe if probe.target_dim == 1 else Dataset(
        storage_env_var="GIFT_EVAL", name=dataset_name, term=term, to_univariate=True,
    )
    ds_key = PRETTY_DATASET_NAMES.get(
        dataset_name.split("/")[0].lower() if "/" in dataset_name else dataset_name.lower(),
        dataset_name.split("/")[0].lower() if "/" in dataset_name else dataset_name.lower(),
    )
    ds_dirname = f"{ds_key}_{str(ds.freq)}_{term}".replace("/", "_")

    forecasts = list(predictor.predict(ds.test_data.input, batch_size=batch_size))
    T = int(ds.prediction_length)
    arr = np.zeros((len(forecasts), len(QUANTILE_LEVELS), T), dtype=np.float32)
    for i, fc in enumerate(forecasts):
        for q_idx, q in enumerate(QUANTILE_LEVELS):
            arr[i, q_idx] = np.asarray(fc.quantile(q), dtype=np.float32)

    out = out_root / model_name / ds_dirname
    out.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out / "test_predictions.npz", predictions=arr)
    return ds_dirname, arr


def verify_predictions_against_bundle(model_name: str, ds_dirname: str,
                                      regenerated: np.ndarray,
                                      atol: float = 1e-3, rtol: float = 1e-3) -> None:
    """Assert a regenerated prediction array matches the bundle's copy.

    Foundation-model inference can have minor non-determinism across hardware
    and library versions, so the comparison uses `np.allclose` with loose
    tolerances rather than strict equality.
    """
    bundle_path = ARTIFACTS_DIR / "test_predictions" / model_name / ds_dirname / "test_predictions.npz"
    with np.load(bundle_path) as npz:
        expected = npz["predictions"]
    if regenerated.shape != expected.shape:
        raise AssertionError(f"shape {regenerated.shape} vs bundle {expected.shape}")
    if not np.allclose(regenerated, expected, atol=atol, rtol=rtol, equal_nan=True):
        max_abs = float(np.max(np.abs(regenerated - expected)))
        raise AssertionError(
            f"{model_name}/{ds_dirname}: max abs diff {max_abs:.4g} exceeds tolerance "
            f"(atol={atol}, rtol={rtol})"
        )
    print(f"  {model_name}/{ds_dirname}: matches bundle "
          f"({len(expected):,} windows, atol={atol}, rtol={rtol})")

#### Per-model predictor factories

Each factory below returns a gluonts predictor configured for one `(dataset, term)`. The bodies are intentionally thin — most of the work is delegated to each base model's own library. See each model's GIFT-Eval notebook in this folder for the full setup (`chronos-2.ipynb`, `timesfm2p5.ipynb`, `flowstate.ipynb`, `tirex.ipynb`, `patchtst_fm.ipynb`, `toto_2_0.ipynb`).

In [ ]:
# Chronos-2 (Amazon)
# pip install chronos-forecasting
# Checkpoint: https://huggingface.co/amazon/chronos-2

def make_chronos2_predictor(prediction_length: int, target_dim: int, device: str = "cuda"):
    import torch
    from chronos import BaseChronosPipeline, Chronos2Pipeline
    from gluonts.model.forecast import QuantileForecast

    pipeline = BaseChronosPipeline.from_pretrained(
        "amazon/chronos-2",
        torch_dtype=torch.bfloat16,
        device_map=device,
    )
    assert isinstance(pipeline, Chronos2Pipeline)

    class _Adapter:
        def predict(self, test_data_input, batch_size: int = 512):
            from gluonts.itertools import batcher
            for batch in batcher(test_data_input, batch_size=batch_size):
                inputs = [{"target": item["target"]} for item in batch]
                q_preds = pipeline.predict_quantiles(
                    inputs, prediction_length=prediction_length,
                    quantile_levels=QUANTILE_LEVELS,
                )
                for q in q_preds:
                    arr = np.asarray(q)  # (Q, T)
                    yield QuantileForecast(
                        forecast_arrays=arr,
                        forecast_keys=[str(q_) for q_ in QUANTILE_LEVELS],
                        start_date=None,
                    )
    return _Adapter()

In [ ]:
# TimesFM-2.5 (Google)
# git clone https://github.com/google-research/timesfm.git && pip install -e timesfm
# Checkpoint: https://huggingface.co/google/timesfm-2.5-200m-pytorch

def make_timesfm25_predictor(prediction_length: int, target_dim: int, device: str = "cuda"):
    from timesfm.timesfm_2p5 import timesfm_2p5_torch
    from gluonts.itertools import batcher
    from gluonts.model.forecast import QuantileForecast

    tfm = timesfm_2p5_torch.TimesFM_2p5_200M_torch()
    tfm.load_checkpoint()

    class _Adapter:
        def predict(self, test_data_input, batch_size: int = 512):
            for batch in batcher(test_data_input, batch_size=batch_size):
                contexts = [np.asarray(item["target"]).ravel() for item in batch]
                _, q = tfm.forecast(
                    inputs=contexts,
                    horizon=prediction_length,
                    quantiles=QUANTILE_LEVELS,
                )  # q: (B, T, Q)
                q = np.asarray(q, dtype=np.float32)
                for i in range(q.shape[0]):
                    yield QuantileForecast(
                        forecast_arrays=q[i].T,  # (Q, T)
                        forecast_keys=[str(q_) for q_ in QUANTILE_LEVELS],
                        start_date=None,
                    )
    return _Adapter()

In [ ]:
# FlowState (IBM)
# git clone -b gift-flowstate git@github.com:ibm-granite/granite-tsfm.git
# pip install "granite-tsfm[notebooks]"
# Checkpoint: https://huggingface.co/ibm-research/flowstate

def make_flowstate_predictor(prediction_length: int, target_dim: int,
                             freq: str, domain: str = None, device: str = "cuda"):
    from tsfm_public import FlowStateForPrediction
    from notebooks.hfdemo.flowstate.gift_wrapper import FlowState_Gift_Wrapper

    flowstate = FlowStateForPrediction.from_pretrained("ibm-research/flowstate").to(device)
    flowstate.config.min_context = 0
    flowstate.config.device = device
    return FlowState_Gift_Wrapper(
        flowstate, prediction_length, n_ch=target_dim, batch_size=64,
        f=freq, device=device, domain=domain,
    )

In [ ]:
# TiRex (NX-AI)
# pip install tirex-ts
# Checkpoint: https://huggingface.co/NX-AI/TiRex-1.1-gifteval

def make_tirex_predictor(prediction_length: int, target_dim: int, device: str = "cuda"):
    from tirex import load_model
    from gluonts.itertools import batcher
    from gluonts.model.forecast import QuantileForecast

    model = load_model("NX-AI/TiRex-1.1-gifteval", device=device)

    class _Adapter:
        def predict(self, test_data_input, batch_size: int = 512):
            for batch in batcher(test_data_input, batch_size=batch_size):
                contexts = [np.asarray(item["target"], dtype=np.float32).ravel() for item in batch]
                q = model.forecast(
                    contexts, prediction_length=prediction_length,
                    quantile_levels=QUANTILE_LEVELS,
                )  # (B, T, Q)
                q = np.asarray(q, dtype=np.float32)
                for i in range(q.shape[0]):
                    yield QuantileForecast(
                        forecast_arrays=q[i].T,
                        forecast_keys=[str(q_) for q_ in QUANTILE_LEVELS],
                        start_date=None,
                    )
    return _Adapter()

In [ ]:
# PatchTST-FM (IBM)
# git clone -b patchtst-fm-gift git@github.com:ibm-granite/granite-tsfm.git
# pip install "granite-tsfm[notebooks]"
# Checkpoint: https://huggingface.co/ibm-research/patchtst-fm-r1

def make_patchtstfm_predictor(prediction_length: int, target_dim: int, device: str = "cuda"):
    import sys
    sys.path.append("./granite-tsfm/notebooks/hfdemo/patchtst_fm/")
    from tsfm_public import PatchTSTFMForPrediction
    from patchtst_fm_predictor import PatchTSTFMEvalPredictor

    model = PatchTSTFMForPrediction.from_pretrained("ibm-research/patchtst-fm-r1", device_map=device)
    return PatchTSTFMEvalPredictor(
        model=model,
        prediction_length=prediction_length,
        quantile_levels=QUANTILE_LEVELS,
    )

In [ ]:
# Toto 2.0 (Datadog) — parameterized by checkpoint size
# pip install "toto-2 @ git+https://github.com/DataDog/toto.git#subdirectory=toto2"
# Checkpoints: https://huggingface.co/Datadog/Toto-2.0-{4m,22m,313m,1b,2.5b}

def make_toto2_predictor(prediction_length: int, target_dim: int,
                         size: str, ds, device: str = "cuda"):
    """`size` is one of {'4m','22m','313m','1b','2.5b'}. `ds` is the gift_eval Dataset
    (used for context_length and past_feat_dynamic_real_dim)."""
    from toto2 import Toto2Model, Toto2GluonTSModel, Toto2GluonTSModelConfig

    checkpoint = f"Datadog/Toto-2.0-{size}"
    model = Toto2Model.from_pretrained(checkpoint, map_location=device).to(device).eval()
    config = Toto2GluonTSModelConfig(
        prediction_length=prediction_length,
        context_length=4096,
        target_dim=target_dim,
        past_feat_dynamic_real_dim=ds.past_feat_dynamic_real_dim,
    )
    gts_model = Toto2GluonTSModel(model, config).to(device).eval()
    return gts_model.create_predictor(batch_size=512, device=device)

#### Smoke test: regenerate predictions on `m4_weekly/W/short` and verify

The cell below tries each factory. Anything not installed is logged and skipped, so the test runs with whatever subset of pool members you have on this machine.

In [ ]:
REGEN_PREDICTIONS_ROOT = Path("./regen/test_predictions")
DEVICE = "cuda" if "torch" in globals() and torch.cuda.is_available() else "cpu"  # type: ignore[name-defined]

probe_ds = Dataset(storage_env_var="GIFT_EVAL", name="m4_weekly", term="short", to_univariate=False)
SMOKE_DS = probe_ds if probe_ds.target_dim == 1 else Dataset(
    storage_env_var="GIFT_EVAL", name="m4_weekly", term="short", to_univariate=True,
)

FACTORIES = [
    ("chronos-2",    lambda: make_chronos2_predictor(SMOKE_DS.prediction_length, SMOKE_DS.target_dim, DEVICE)),
    ("timesfm-2.5",  lambda: make_timesfm25_predictor(SMOKE_DS.prediction_length, SMOKE_DS.target_dim, DEVICE)),
    ("flowstate",    lambda: make_flowstate_predictor(SMOKE_DS.prediction_length, SMOKE_DS.target_dim, str(SMOKE_DS.freq), domain=None, device=DEVICE)),
    ("tirex",        lambda: make_tirex_predictor(SMOKE_DS.prediction_length, SMOKE_DS.target_dim, DEVICE)),
    ("patchtst-fm",  lambda: make_patchtstfm_predictor(SMOKE_DS.prediction_length, SMOKE_DS.target_dim, DEVICE)),
    ("toto-2.0-4m",  lambda: make_toto2_predictor(SMOKE_DS.prediction_length, SMOKE_DS.target_dim, "4m",   SMOKE_DS, DEVICE)),
    ("toto-2.0-22m", lambda: make_toto2_predictor(SMOKE_DS.prediction_length, SMOKE_DS.target_dim, "22m",  SMOKE_DS, DEVICE)),
    ("toto-2.0-313m",lambda: make_toto2_predictor(SMOKE_DS.prediction_length, SMOKE_DS.target_dim, "313m", SMOKE_DS, DEVICE)),
    ("toto-2.0-1b",  lambda: make_toto2_predictor(SMOKE_DS.prediction_length, SMOKE_DS.target_dim, "1b",   SMOKE_DS, DEVICE)),
    ("toto-2.0-2.5b",lambda: make_toto2_predictor(SMOKE_DS.prediction_length, SMOKE_DS.target_dim, "2.5b", SMOKE_DS, DEVICE)),
]

for model_name, factory in FACTORIES:
    try:
        predictor = factory()
    except Exception as e:
        print(f"  {model_name}: skipped ({type(e).__name__}: {str(e)[:80]})")
        continue
    try:
        ds_dirname, arr = regenerate_predictions(
            predictor, model_name, "m4_weekly", "short", REGEN_PREDICTIONS_ROOT,
        )
        verify_predictions_against_bundle(model_name, ds_dirname, arr)
    except Exception as e:
        print(f"  {model_name}: FAILED ({type(e).__name__}: {str(e)[:120]})")

### Optional C. Upload your regenerated artifacts to Hugging Face Hub

If you've regenerated the artifacts (e.g. for a different base-model pool) and want a reproducible bundle for others, push them to a HF model repo. The same `snapshot_download` call in Section 1 will then pull from your repo.

In [ ]:
# pip install huggingface_hub
# huggingface-cli login
from huggingface_hub import HfApi

# api = HfApi()
# api.create_repo(repo_id=HF_ARTIFACTS_REPO, repo_type="model", private=True, exist_ok=True)
# api.upload_folder(
#     folder_path=str(ARTIFACTS_DIR),
#     repo_id=HF_ARTIFACTS_REPO,
#     repo_type="model",
#     commit_message="Toto 2.0 Family and Friends GIFT-Eval artifact bundle",
# )